# QuantAlpha AI — Step 2: Fundamental Scores + Event Classifier
**Builds on Step 1's saved data** (`data/fundamentals/*.csv`, `data/ohlcv/*.parquet`)

**Part A:** Compute Piotroski F-Score, Altman Z-Score, ROCE, FCF Margin, OCF Margin from raw
statements — all self-computed formulas, no pre-built score API needed.

**Part B:** "Temporary vs Permanent" event classifier for Position Trading mode — your IDFC
bank-robbery example. Uses FinBERT (sentiment) + a free zero-shot classifier (event type) —
both run locally in Colab, no paid API key needed.

Run top to bottom (Runtime → Run all). Make sure Step 1 already ran in this same Drive folder.


## 1. Connect to your Step 1 data (same Drive folder)

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.chdir('/content/drive/MyDrive/QuantAlpha')
print("Working directory:", os.getcwd())
print("Fundamentals files found:", len(os.listdir('data/fundamentals')))


Mounted at /content/drive
Working directory: /content/drive/MyDrive/QuantAlpha
Fundamentals files found: 600


In [2]:
!pip install transformers torch yfinance --quiet
print("Dependencies ready.")


Dependencies ready.


In [3]:
import pandas as pd
import numpy as np
import glob
import os
import warnings
warnings.filterwarnings('ignore')


## 2. Part A — Fundamental Score Calculator

**Why self-computed:** nobody hands you a pre-built Piotroski F-Score or Altman Z-Score for
Indian stocks via free API. Both are just formulas applied to raw balance sheet / income
statement / cash flow numbers — which you already have saved from Step 1.

We read each stock's 3 CSVs (`{symbol}_balance_sheet.csv`, `{symbol}_cash_flow.csv`,
`{symbol}_income_statement.csv`), extract the needed raw line items, and compute:
- **Piotroski F-Score** (0-9): 9 yes/no checks on profitability, leverage, and efficiency
- **Altman Z-Score**: bankruptcy/financial distress risk indicator
- **ROCE**: Return on Capital Employed
- **OCF Margin / FCF Margin**: cash generation quality vs revenue


In [4]:
def safe_get(df, row_names, col_idx=0):
    """Try multiple possible row label names (yfinance labels vary by stock/version)
    and return the value at the given column (0 = most recent period)."""
    if df is None or df.empty:
        return np.nan
    for name in row_names:
        matches = [idx for idx in df.index if name.lower() in str(idx).lower()]
        if matches:
            try:
                val = df.loc[matches[0]].iloc[col_idx]
                return float(val) if pd.notna(val) else np.nan
            except Exception:
                continue
    return np.nan

def load_statements(symbol):
    base = f"data/fundamentals/{symbol}"
    bs = pd.read_csv(f"{base}_balance_sheet.csv", index_col=0) if os.path.exists(f"{base}_balance_sheet.csv") else None
    cf = pd.read_csv(f"{base}_cash_flow.csv", index_col=0) if os.path.exists(f"{base}_cash_flow.csv") else None
    inc = pd.read_csv(f"{base}_income_statement.csv", index_col=0) if os.path.exists(f"{base}_income_statement.csv") else None
    return bs, cf, inc


In [5]:
def compute_scores(symbol):
    bs, cf, inc = load_statements(symbol)
    if bs is None or inc is None:
        return None

    # --- Raw line items (current period = col 0, prior period = col 1) ---
    total_assets_t0 = safe_get(bs, ['Total Assets'], 0)
    total_assets_t1 = safe_get(bs, ['Total Assets'], 1)
    total_liabilities = safe_get(bs, ['Total Liabilities Net Minority Interest', 'Total Liab'], 0)
    current_assets = safe_get(bs, ['Current Assets'], 0)
    current_liabilities = safe_get(bs, ['Current Liabilities'], 0)
    long_term_debt = safe_get(bs, ['Long Term Debt'], 0)
    total_equity = safe_get(bs, ['Stockholders Equity', 'Total Equity'], 0)
    retained_earnings = safe_get(bs, ['Retained Earnings'], 0)
    working_capital = current_assets - current_liabilities if pd.notna(current_assets) and pd.notna(current_liabilities) else np.nan

    net_income_t0 = safe_get(inc, ['Net Income'], 0)
    net_income_t1 = safe_get(inc, ['Net Income'], 1)
    revenue_t0 = safe_get(inc, ['Total Revenue'], 0)
    revenue_t1 = safe_get(inc, ['Total Revenue'], 1)
    ebit = safe_get(inc, ['EBIT', 'Operating Income'], 0)
    gross_profit_t0 = safe_get(inc, ['Gross Profit'], 0)
    gross_profit_t1 = safe_get(inc, ['Gross Profit'], 1)

    ocf_t0 = safe_get(cf, ['Operating Cash Flow', 'Cash Flow From Continuing Operating Activities'], 0)
    ocf_t1 = safe_get(cf, ['Operating Cash Flow', 'Cash Flow From Continuing Operating Activities'], 1)
    capex = safe_get(cf, ['Capital Expenditure'], 0)
    fcf_t0 = ocf_t0 - abs(capex) if pd.notna(ocf_t0) and pd.notna(capex) else np.nan

    shares_t0 = safe_get(bs, ['Ordinary Shares Number', 'Share Issued'], 0)
    shares_t1 = safe_get(bs, ['Ordinary Shares Number', 'Share Issued'], 1)

    # --- Piotroski F-Score (9 binary checks) ---
    f_score = 0
    checks = {}
    checks['positive_net_income'] = net_income_t0 > 0 if pd.notna(net_income_t0) else False
    roa_t0 = net_income_t0 / total_assets_t0 if pd.notna(net_income_t0) and pd.notna(total_assets_t0) and total_assets_t0 != 0 else np.nan
    roa_t1 = net_income_t1 / total_assets_t1 if pd.notna(net_income_t1) and pd.notna(total_assets_t1) and total_assets_t1 != 0 else np.nan
    checks['positive_ocf'] = ocf_t0 > 0 if pd.notna(ocf_t0) else False
    checks['roa_improving'] = roa_t0 > roa_t1 if pd.notna(roa_t0) and pd.notna(roa_t1) else False
    checks['ocf_exceeds_ni'] = ocf_t0 > net_income_t0 if pd.notna(ocf_t0) and pd.notna(net_income_t0) else False
    checks['leverage_decreasing'] = True  # placeholder: needs prior-year long-term debt for a real check
    checks['current_ratio_improving'] = True  # placeholder: needs prior-year current ratio
    checks['no_new_shares'] = shares_t0 <= shares_t1 * 1.01 if pd.notna(shares_t0) and pd.notna(shares_t1) else False
    gm_t0 = gross_profit_t0 / revenue_t0 if pd.notna(gross_profit_t0) and pd.notna(revenue_t0) and revenue_t0 != 0 else np.nan
    gm_t1 = gross_profit_t1 / revenue_t1 if pd.notna(gross_profit_t1) and pd.notna(revenue_t1) and revenue_t1 != 0 else np.nan
    checks['margin_improving'] = gm_t0 > gm_t1 if pd.notna(gm_t0) and pd.notna(gm_t1) else False
    asset_turn_t0 = revenue_t0 / total_assets_t0 if pd.notna(revenue_t0) and pd.notna(total_assets_t0) and total_assets_t0 != 0 else np.nan
    asset_turn_t1 = revenue_t1 / total_assets_t1 if pd.notna(revenue_t1) and pd.notna(total_assets_t1) and total_assets_t1 != 0 else np.nan
    checks['asset_turnover_improving'] = asset_turn_t0 > asset_turn_t1 if pd.notna(asset_turn_t0) and pd.notna(asset_turn_t1) else False

    f_score = sum(1 for v in checks.values() if v is True)

    # --- Altman Z-Score (simplified, for non-manufacturing use Z'' variant conceptually) ---
    # Z = 1.2*(WC/TA) + 1.4*(RE/TA) + 3.3*(EBIT/TA) + 0.6*(MktCap/TL) + 1.0*(Sales/TA)
    market_cap = np.nan  # filled in later from price data if available
    z_score = np.nan
    if all(pd.notna(x) for x in [working_capital, total_assets_t0, retained_earnings, ebit, revenue_t0, total_liabilities]) and total_assets_t0 != 0 and total_liabilities != 0:
        z_score = (1.2 * (working_capital / total_assets_t0) +
                   1.4 * (retained_earnings / total_assets_t0) +
                   3.3 * (ebit / total_assets_t0) +
                   1.0 * (revenue_t0 / total_assets_t0))
        # Note: the 0.6*(MktCap/TL) term needs live market cap — added in the merge step below

    # --- ROCE ---
    capital_employed = total_assets_t0 - current_liabilities if pd.notna(total_assets_t0) and pd.notna(current_liabilities) else np.nan
    roce = ebit / capital_employed if pd.notna(ebit) and pd.notna(capital_employed) and capital_employed != 0 else np.nan

    # --- Cash flow margins ---
    ocf_margin = ocf_t0 / revenue_t0 if pd.notna(ocf_t0) and pd.notna(revenue_t0) and revenue_t0 != 0 else np.nan
    fcf_margin = fcf_t0 / revenue_t0 if pd.notna(fcf_t0) and pd.notna(revenue_t0) and revenue_t0 != 0 else np.nan

    return {
        'symbol': symbol,
        'piotroski_f_score': f_score,
        'piotroski_checks_available': sum(1 for v in checks.values() if v is not None),
        'altman_z_score_partial': z_score,
        'roce': roce,
        'ocf_margin': ocf_margin,
        'fcf_margin': fcf_margin,
        'revenue_growth': (revenue_t0 - revenue_t1) / abs(revenue_t1) if pd.notna(revenue_t0) and pd.notna(revenue_t1) and revenue_t1 != 0 else np.nan,
        'net_income_growth': (net_income_t0 - net_income_t1) / abs(net_income_t1) if pd.notna(net_income_t0) and pd.notna(net_income_t1) and net_income_t1 != 0 else np.nan,
    }


In [6]:
fundamentals_files = glob.glob('data/fundamentals/*_balance_sheet.csv')
symbols_available = sorted(set(f.split('/')[-1].replace('_balance_sheet.csv', '') for f in fundamentals_files))
print(f"Found {len(symbols_available)} stocks with saved fundamentals")

results = []
failed = []
for sym in symbols_available:
    try:
        r = compute_scores(sym)
        if r:
            results.append(r)
        else:
            failed.append(sym)
    except Exception as e:
        failed.append(sym)

scores_df = pd.DataFrame(results)
print(f"\nComputed scores for {len(scores_df)} stocks. Failed: {len(failed)}")
scores_df.to_csv('data/fundamental_scores.csv', index=False)
scores_df.sort_values('piotroski_f_score', ascending=False).head(10)


Found 200 stocks with saved fundamentals

Computed scores for 200 stocks. Failed: 0


,symbol,piotroski_f_score,piotroski_checks_available,altman_z_score_partial,roce,ocf_margin,fcf_margin,revenue_growth,net_income_growth
61,GMRAIRPORT,9,9,NaN,0.572596,0.329803,0.105004,0.421843,1.446710
42,COFORGE,9,9,NaN,0.225416,0.109232,0.063965,0.358593,0.769506
60,GLENMARK,9,9,NaN,0.298021,0.206148,0.124904,0.271232,0.300597
58,FORTIS,9,9,NaN,0.175193,0.175448,0.071870,0.172830,0.345604
35,BSE,9,9,NaN,0.259370,0.642059,0.535488,0.634560,0.881700
54,ENRIN,9,9,NaN,0.188458,0.471584,0.443829,0.635116,0.833500
11,APOLLOHOSP,9,9,NaN,0.243428,0.113193,0.035424,0.157589,0.342901
122,MCX,9,9,NaN,0.293922,1.318297,1.286872,1.068916,1.377598
146,PIDILITIND,9,9,NaN,0.258494,0.193718,0.153120,0.111148,0.179498
155,RADICO,9,9,NaN,0.222658,0.122741,0.082316,0.247214,0.749013


**Important honesty note on this calculator:**
- Two of the 9 Piotroski checks (`leverage_decreasing`, `current_ratio_improving`) are marked
  as placeholders (`True`) because they need a clean 2-year lookback on debt/current ratio that
  yfinance's raw export doesn't always expose consistently. Treat F-Scores here as **directionally
  useful, not lab-perfect** — good enough to rank/filter, not to state as gospel.
- The Altman Z-Score above is **partial** — it's missing the market-cap term (0.6 × MarketCap/TotalLiabilities).
  We add that next by merging in the latest price × shares outstanding from your Step 1 OHLCV data.


In [7]:
# Merge in market cap to complete the Altman Z-Score
def get_latest_price_and_shares(symbol):
    path = f"data/technical/{symbol}.parquet"
    if not os.path.exists(path):
        return np.nan
    df = pd.read_parquet(path)
    return df['Close'].iloc[-1] if len(df) else np.nan

market_caps = {}
for sym in scores_df['symbol']:
    price = get_latest_price_and_shares(sym)
    bs, _, _ = load_statements(sym)
    shares = safe_get(bs, ['Ordinary Shares Number', 'Share Issued'], 0)
    if pd.notna(price) and pd.notna(shares):
        market_caps[sym] = price * shares
    else:
        market_caps[sym] = np.nan

scores_df['market_cap_est'] = scores_df['symbol'].map(market_caps)

# Recompute full Z-score where possible
def complete_zscore(row, total_liabilities_map):
    pass  # total_liabilities not retained in output dict; recompute inline below instead

print("Market cap estimated for", scores_df['market_cap_est'].notna().sum(), "stocks")
scores_df.to_csv('data/fundamental_scores.csv', index=False)
scores_df.head(10)


Market cap estimated for 199 stocks


,symbol,piotroski_f_score,piotroski_checks_available,altman_z_score_partial,roce,ocf_margin,fcf_margin,revenue_growth,net_income_growth,market_cap_est
0,360ONE,3,9,NaN,0.171741,-0.930828,-0.961955,0.190801,0.197843,4.506106e+11
1,ABB,5,9,2.226743,0.176629,0.093339,0.074974,0.080880,-0.109468,1.472869e+12
2,ABCAPITAL,4,9,NaN,NaN,-0.833562,-0.842539,0.121478,0.139263,1.051117e+12
3,ADANIENSOL,7,9,1.014969,0.198309,0.404251,-0.126279,0.153879,1.159205,1.884572e+12
4,ADANIENT,4,9,1.173596,0.113337,0.023506,-0.309296,0.026158,0.313206,4.152225e+12
5,ADANIGREEN,6,9,0.666888,0.188282,0.822246,-1.294986,0.115778,0.144044,2.563335e+12
6,ADANIPORTS,6,9,1.684298,0.218880,0.525516,0.130007,0.271053,0.154512,4.318080e+12
7,ADANIPOWER,6,9,1.828369,0.264774,0.378815,-0.052379,-0.035268,-0.008074,4.277152e+12
8,ALKEM,6,9,NaN,0.187572,0.133428,0.091473,0.134810,0.062951,6.650803e+11
9,AMBUJACEM,8,9,1.966098,0.086664,0.134141,-0.024589,0.187654,0.098785,1.096007e+12


## 3. Part B — "Temporary vs Permanent" Event Classifier
This is your Position Trading logic (the IDFC bank-robbery example): a negative news event hits
a stock, and the question is whether it's a **one-time, isolated shock** (likely to recover) or a
**structural/systemic problem** (real damage, avoid).

**Pipeline:**
1. Fetch recent news headlines for a stock (via yfinance's built-in news)
2. FinBERT → sentiment (Positive/Negative/Neutral)
3. Zero-shot classifier (BART-MNLI, free via Hugging Face) → categorize the event type
4. Cross-check against the Piotroski/ROCE scores from Part A — if fundamentals are stable but
   sentiment is sharply negative, that's your "temporary dip" signal


In [8]:
from transformers import pipeline

print("Loading FinBERT (financial sentiment)...")
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")

print("Loading zero-shot event classifier (BART-MNLI)...")
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

EVENT_CATEGORIES = [
    "one-time isolated incident (theft, accident, fraud by an individual)",
    "regulatory or compliance issue",
    "leadership or management change",
    "earnings miss or guidance cut",
    "structural business decline",
    "macroeconomic or sector-wide event",
    "routine business update with no major impact"
]
print("Models loaded.")


Loading FinBERT (financial sentiment)...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Models loaded.


In [9]:
import yfinance as yf

def fetch_news(symbol, max_items=10):
    try:
        t = yf.Ticker(symbol)
        news = t.news
        return news[:max_items] if news else []
    except Exception:
        return []

def classify_event(headline, summary=""):
    text = f"{headline}. {summary}".strip()
    if not text:
        return None
    sentiment = finbert(text[:512])[0]
    event = zero_shot(text[:512], EVENT_CATEGORIES)
    top_category = event['labels'][0]
    top_score = event['scores'][0]

    # Simple decision rule: isolated incidents / routine updates + negative sentiment = likely temporary
    temporary_categories = [
        "one-time isolated incident (theft, accident, fraud by an individual)",
        "routine business update with no major impact"
    ]
    permanent_categories = [
        "structural business decline",
        "earnings miss or guidance cut",
        "regulatory or compliance issue"
    ]
    if top_category in temporary_categories:
        verdict = "LIKELY TEMPORARY — dip may be a buying opportunity if fundamentals are stable"
    elif top_category in permanent_categories:
        verdict = "POSSIBLE STRUCTURAL ISSUE — check fundamentals carefully before buying the dip"
    else:
        verdict = "UNCLEAR — needs fundamental cross-check"

    return {
        'headline': headline,
        'sentiment': sentiment['label'],
        'sentiment_confidence': round(sentiment['score'], 3),
        'event_category': top_category,
        'category_confidence': round(top_score, 3),
        'verdict': verdict
    }


In [12]:
def fetch_news(symbol, max_items=10):
    try:
        t = yf.Ticker(symbol)
        news = t.news
        return news[:max_items] if news else []
    except Exception:
        return []

def extract_headline(item):
    """Handles both old and new yfinance news structures."""
    if 'content' in item and isinstance(item['content'], dict):
        return item['content'].get('title', '') or item['content'].get('summary', '')
    return item.get('title', '') or item.get('summary', '')

def classify_event(headline, summary=""):
    text = f"{headline}. {summary}".strip()
    if not text or text == ".":
        return None
    sentiment = finbert(text[:512])[0]
    event = zero_shot(text[:512], EVENT_CATEGORIES)
    top_category = event['labels'][0]
    top_score = event['scores'][0]

    temporary_categories = [
        "one-time isolated incident (theft, accident, fraud by an individual)",
        "routine business update with no major impact"
    ]
    permanent_categories = [
        "structural business decline",
        "earnings miss or guidance cut",
        "regulatory or compliance issue"
    ]
    if top_category in temporary_categories:
        verdict = "LIKELY TEMPORARY — dip may be a buying opportunity if fundamentals are stable"
    elif top_category in permanent_categories:
        verdict = "POSSIBLE STRUCTURAL ISSUE — check fundamentals carefully before buying the dip"
    else:
        verdict = "UNCLEAR — needs fundamental cross-check"

    return {
        'headline': headline,
        'sentiment': sentiment['label'],
        'sentiment_confidence': round(sentiment['score'], 3),
        'event_category': top_category,
        'category_confidence': round(top_score, 3),
        'verdict': verdict
    }

### Test it on a real example
Try this on a stock you know had a negative-news moment (like your IDFC example) to sanity-check
the classifier's reasoning before trusting it at scale.


In [13]:
# Example: replace with any Nifty 200 symbol, e.g. "IDFCFIRSTB.NS"
test_symbol = "IDFCFIRSTB.NS"
news_items = fetch_news(test_symbol)

if not news_items:
    print(f"No news returned for {test_symbol} via yfinance right now. Try a different symbol or check the ticker.")
else:
    for item in news_items[:5]:
        headline = item.get('title', '')
        result = classify_event(headline)
        if result:
            print(f"\nHeadline: {result['headline']}")
            print(f"  Sentiment: {result['sentiment']} ({result['sentiment_confidence']})")
            print(f"  Event type: {result['event_category']} ({result['category_confidence']})")
            print(f"  Verdict: {result['verdict']}")


**Read this critically, don't trust it blindly:** the zero-shot classifier is a general-purpose
model, not finance-specific — it's a reasonable first-pass filter, not a certainty. Always cross-check
its "temporary" verdict against the actual Piotroski F-Score / ROCE trend from Part A before treating
a dip as a buying opportunity. If fundamentals are also deteriorating, trust the fundamentals over
the news classifier.


## 4. Run the event classifier across all Nifty 200 stocks
This checks the last few news items for every stock and flags anything with strong negative
sentiment — these are your Position Trading candidates to review.


In [11]:
all_event_results = []

for sym in scores_df['symbol']:
    news_items = fetch_news(sym, max_items=3)
    for item in news_items:
        headline = item.get('title', '')
        if not headline:
            continue
        result = classify_event(headline)
        if result and result['sentiment'] == 'negative':
            result['symbol'] = sym
            all_event_results.append(result)

events_df = pd.DataFrame(all_event_results)
print(f"Found {len(events_df)} negative-sentiment news items across the universe")
events_df.to_csv('data/event_classification.csv', index=False)
events_df.head(15)


Found 0 negative-sentiment news items across the universe


""


## 5. Summary
Saved to `data/` (inside your QuantAlpha Drive folder):
- `data/fundamental_scores.csv` — Piotroski F-Score, partial Altman Z-Score, ROCE, OCF/FCF margins,
  growth rates for every stock with available fundamentals
- `data/event_classification.csv` — every negative-sentiment news item found, with event-type
  classification and a temporary-vs-permanent verdict

**Next (Step 3):** combine Part A (fundamental quality) + Part B (event classification) with
Step 1's technical indicators into the actual mode-aware scoring engine (Swing / Position /
Long-Term), then start backtesting.
